# SuperTrend 交易績效追蹤與回測系統
---
本 Notebook 提供 SuperTrend 交易策略的互動式分析介面。系統支援：
1. **SQLite 本地快取**：自動儲存 Yahoo Finance 歷史資料於本地資料庫中，減少重複請求。
2. **技術指標計算**：計算 SuperTrend, Hull Moving Average (HMA), ADX, RSI。
3. **精確交易回測**：扣除手續費/滑價，處理開盤價跳空止損，支援固定風險比例部位控制。
4. **績效指標評估**：計算 CAGR、最大回撤、夏普比率、索提諾比率、交易期望值與 **Ulcer Index (潰瘍指數)**。
5. **可重現 Monte Carlo 模擬**：分析交易回報序列，展示 95% 及 99% 最壞風險路徑分位數。
6. **參數熱力圖尋優**：藉由 Walk-Forward 劃分尋找參數穩健高原區 (Plateau)。

## 使用流程
1. **Cell 2**：設定標的、日期區間、K 線週期
2. **Cell 3**：執行參數優化，取得最佳 SuperTrend 參數
3. **Cell 4**：使用最佳參數（或自訂參數）執行回測
4. **Cell 5-9**：檢視圖表、績效指標、風險分析、完整交易明細

In [ ]:
# Cell 1: 載入核心模組與依賴庫
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_loader import DataLoader
from src.indicators import compute_supertrend, compute_hma, compute_adx, compute_rsi
from src.backtest_engine import BacktestEngine
from src.risk_manager import RiskManager
from src.optimizer import ParameterOptimizer

print("模組載入成功！")

In [ ]:
# Cell 2: 設定標的與下載歷史價格資料
# --- 請修改以下參數 ---
ticker = "VT"
start_date = "2018-07-20"
end_date = "2026-07-17"
interval = "1d"  # "1d" 日線, "1h" 小時線

loader = DataLoader()
# 優先從本地 SQLite 讀取，缺失時才會下載並快取
df_raw = loader.fetch_data(ticker, start_date=start_date, end_date=end_date, interval=interval, refresh=False)
print(f"成功加載 {ticker} K 線資料：共 {len(df_raw)} 筆記錄，區間 {df_raw.index[0].date()} ~ {df_raw.index[-1].date()}")
df_raw.tail()

In [ ]:
# Cell 3: 參數優化 - 尋找最佳 SuperTrend 參數 (含 HMA 週期自動優化)
# --- 請修改優化搜尋範圍 ---
st_periods = [5, 10, 15, 20, 25, 30]          # SuperTrend 長度搜尋範圍
st_multipliers = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0]         # SuperTrend 倍數搜尋範圍
ma_periods = [0, 25, 50, 75, 100, 125]                # HMA 過濾器週期搜尋範圍
                                                # 0 = 不使用 HMA 過濾 (模式 1: 需 Crossover)
                                                # >0 = 使用 HMA(ma_p) 過濾 (模式 2: 趨勢跟隨，不需 Crossover)
train_ratio = 0.70                            # 訓練樣本比例 (1.0 = 不切分, 全樣本優化)
target_metric = "Custom Score"                # 優化目標: "Custom Score", "CAGR", "Sharpe Ratio", "Calmar Ratio", "Max Drawdown"

print("正在執行網格優化掃描，這將耗時約 5-10 秒...")
optimizer = ParameterOptimizer(df_raw)

train_res, test_res = optimizer.optimize(
    st_periods=st_periods,
    st_multipliers=st_multipliers,
    ma_periods=ma_periods,
    target_metric=target_metric,
    train_ratio=train_ratio
)

print("\n=== 樣本內 (Train) 前五名最穩健參數 ===")
# 顯示關鍵欄位
display_cols = ["ST_Period", "ST_Multiplier", "MA_Period", "Use_HMA_Filter", "Custom Score", 
                "CAGR", "Max Drawdown", "Sharpe Ratio", "Calmar Ratio", "Win Rate", "Profit Factor"]
display(train_res[display_cols].head(5))

if not test_res.empty:
    print("\n=== 樣本外 (Test) 前五名參數 ===")
    display(test_res[display_cols].head(5))

# 繪製參數熱力圖 - 依 HMA 模式分開顯示
print("\n=== 參數高原熱力圖 ===")

# 分離兩種模式
train_no_hma = train_res[train_res["MA_Period"] == 0]
train_with_hma = train_res[train_res["MA_Period"] > 0]

if not train_no_hma.empty:
    print("--- 模式 1: 無 HMA 過濾 (需 Crossover) ---")
    heatmap_df = train_no_hma.pivot_table(index='ST_Multiplier', columns='ST_Period', values='Custom Score', aggfunc='mean')
    fig_hm = go.Figure(data=go.Heatmap(
        z=heatmap_df.values,
        x=heatmap_df.columns.astype(str),
        y=heatmap_df.index.astype(str),
        colorscale='Viridis',
        colorbar=dict(title='Score'),
        text=np.round(heatmap_df.values, 3),
        texttemplate="%{text}",
        textfont=dict(size=10)
    ))
    fig_hm.update_layout(title='模式 1: 無 HMA 過濾 - 參數高原熱力圖', xaxis_title='SuperTrend Period', yaxis_title='Multiplier', height=400)
    fig_hm.show()

if not train_with_hma.empty:
    print("--- 模式 2: 有 HMA 過濾 (趨勢跟隨，不需 Crossover) ---")
    # 對每個 HMA 週期分別畫熱力圖
    for ma_p in sorted(train_with_hma["MA_Period"].unique()):
        subset = train_with_hma[train_with_hma["MA_Period"] == ma_p]
        heatmap_df = subset.pivot_table(index='ST_Multiplier', columns='ST_Period', values='Custom Score', aggfunc='mean')
        fig_hm = go.Figure(data=go.Heatmap(
            z=heatmap_df.values,
            x=heatmap_df.columns.astype(str),
            y=heatmap_df.index.astype(str),
            colorscale='Viridis',
            colorbar=dict(title='Score'),
            text=np.round(heatmap_df.values, 3),
            texttemplate="%{text}",
            textfont=dict(size=10)
        ))
        fig_hm.update_layout(title=f'模式 2: HMA({ma_p}) 過濾 - 參數高原熱力圖', xaxis_title='SuperTrend Period', yaxis_title='Multiplier', height=400)
        fig_hm.show()

# 取得最佳參數供後續 Cell 使用
best_period = int(train_res.iloc[0]['ST_Period'])
best_multiplier = float(train_res.iloc[0]['ST_Multiplier'])
best_ma_period = int(train_res.iloc[0]['MA_Period'])
best_use_hma = bool(train_res.iloc[0]['Use_HMA_Filter'])

print(f"\n>>> 最佳參數: ST({best_period}, {best_multiplier})")
print(f">>> HMA 過濾: {'啟用 (週期=' + str(best_ma_period) + ')' if best_use_hma else '未啟用 (模式 1: 需 Crossover)'}")
print(f">>> 請在 Cell 4 確認或修改這些參數後再執行回測")

In [ ]:
# Cell 4: 使用最佳參數執行回測
# --- 可選擇使用優化出的最佳參數，或手動指定參數 ---
use_optimized_params = True  # True: 使用 Cell 3 的最佳參數, False: 使用下方手動參數

if use_optimized_params:
    # 從 Cell 3 自動取得最佳參數
    period = best_period
    multiplier = best_multiplier
    ma_period = best_ma_period
    use_hma_filter = best_use_hma
else:
    # 手動指定參數
    period = 25
    multiplier = 2.5
    ma_period = 50          # 0 = 不使用 HMA 過濾; >0 = 使用 HMA(ma_p) 過濾
    use_hma_filter = True   # True = 啟用 HMA 過濾 (模式 2: 趨勢跟隨，不需 Crossover)

# 其他過濾器
use_adx_filter = False  # ADX 趨勢強度過濾
use_rsi_filter = False  # RSI 過濾

print(f"使用參數: ST({period}, {multiplier}), HMA={ma_period}")
print(f"HMA 過濾模式: {'啟用 (模式 2: 趨勢跟隨，不需 Crossover)' if use_hma_filter else '未啟用 (模式 1: 需 Crossover)'}")
print(f"其他過濾器: ADX={use_adx_filter}, RSI={use_rsi_filter}")

# 1. 計算 SuperTrend
df_indicators, trend = compute_supertrend(df_raw, period=period, multiplier=multiplier)

# 2. 計算過濾器指標
if ma_period > 0:
    df_indicators["HMA"] = compute_hma(df_indicators["Close"], period=ma_period)
    # use_hma_filter=True 時：MA_Filter = Close > HMA (有實際過濾效果)
    # use_hma_filter=False 時：MA_Filter = True (不啟用過濾)
    df_indicators["MA_Filter"] = df_indicators["Close"] > df_indicators["HMA"] if use_hma_filter else True
else:
    df_indicators["MA_Filter"] = True

df_indicators["ADX"] = compute_adx(df_indicators, period=14)
df_indicators["ADX_Filter"] = df_indicators["ADX"] > 20 if use_adx_filter else True

df_indicators["RSI"] = compute_rsi(df_indicators["Close"], period=14)
df_indicators["RSI_Filter"] = (df_indicators["RSI"] > 30) & (df_indicators["RSI"] < 70) if use_rsi_filter else True

# 3. 執行回測
engine = BacktestEngine(df_indicators)
trades, strat_equity, bench_equity = engine.run(
    long_only=True,
    stop_loss_pct=0.0,      # 固定停損百分比 (0.0 = 不使用)
    take_profit_pct=0.0,    # 固定停利百分比 (0.0 = 不使用)
    commission_bps=10.0,    # 單邊 10 bps 手續費
    slippage_bps=5.0,       # 單邊 5 bps 滑價
    position_sizing_mode="fixed_fraction"
)

print(f"回測完成！已平倉交易筆數: {len(trades)}")

In [ ]:
# Cell 5: 價格 K 線、SuperTrend 通道與買賣標記圖
fig_price = go.Figure()

# K 線圖
fig_price.add_trace(go.Candlestick(
    x=df_indicators.index,
    open=df_indicators['Open'],
    high=df_indicators['High'],
    low=df_indicators['Low'],
    close=df_indicators['Close'],
    name='Price'
))

# SuperTrend 線
last_trend = df_indicators['Trend'].iloc[-1]
st_color = 'rgba(0, 180, 0, 0.7)' if last_trend == 1 else 'rgba(180, 0, 0, 0.7)'
fig_price.add_trace(go.Scatter(
    x=df_indicators.index,
    y=df_indicators['SuperTrend'],
    line=dict(color=st_color, width=1.5),
    name='SuperTrend'
))

# HMA 線 (若啟用)
# Cell 4 定義的變數為 use_hma_filter
if "HMA" in df_indicators.columns and use_hma_filter:
    fig_price.add_trace(go.Scatter(
        x=df_indicators.index,
        y=df_indicators['HMA'],
        line=dict(color='rgba(0, 0, 200, 0.5)', width=1, dash='dot'),
        name=f'HMA({ma_period})'
    ))

# 標記交易進出場點
buy_dates = [t.entry_date for t in trades]
buy_prices = [t.entry_price for t in trades]
sell_dates = [t.exit_date for t in trades]
sell_prices = [t.exit_price for t in trades]

fig_price.add_trace(go.Scatter(
    x=buy_dates, y=buy_prices, mode='markers',
    marker=dict(symbol='triangle-up', size=12, color='green', line=dict(width=1, color='black')),
    name='Buy (Entry)',
    hovertemplate='買入: %{x|%Y-%m-%d}<br>價格: $%{y:.2f}<extra></extra>'
))

fig_price.add_trace(go.Scatter(
    x=sell_dates, y=sell_prices, mode='markers',
    marker=dict(symbol='triangle-down', size=12, color='red', line=dict(width=1, color='black')),
    name='Sell (Exit)',
    hovertemplate='賣出: %{x|%Y-%m-%d}<br>價格: $%{y:.2f}<extra></extra>'
))

fig_price.update_layout(
    title=f'{ticker} SuperTrend({period}, {multiplier}) 策略交易點標記圖',
    xaxis_rangeslider_visible=False,
    height=600,
    hovermode='x unified'
)
fig_price.show()

In [ ]:
# Cell 6: 策略與 Benchmark 淨值曲線與回撤軌跡對比圖
roll_max_strat = strat_equity.cummax()
dd_strat = (strat_equity - roll_max_strat) / roll_max_strat

roll_max_bench = bench_equity.cummax()
dd_bench = (bench_equity - roll_max_bench) / roll_max_bench

fig_perf = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])

# 淨值對比
fig_perf.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity, name='SuperTrend Strategy', line=dict(color='blue', width=2)), row=1, col=1)
fig_perf.add_trace(go.Scatter(x=bench_equity.index, y=bench_equity, name='Buy & Hold Benchmark', line=dict(color='gray', width=1.5, dash='dot')), row=1, col=1)

# 回撤軌跡
fig_perf.add_trace(go.Scatter(x=dd_strat.index, y=dd_strat * 100, name='Strategy Drawdown', fill='tozeroy', line=dict(color='rgba(255, 0, 0, 0.5)', width=1)), row=2, col=1)
fig_perf.add_trace(go.Scatter(x=dd_bench.index, y=dd_bench * 100, name='Benchmark Drawdown', line=dict(color='gray', width=1)), row=2, col=1)

fig_perf.update_layout(title='績效與回撤軌跡對比圖 (Equity & Drawdown Path)', height=600)
fig_perf.update_yaxes(title_text='Portfolio Value ($)', row=1, col=1)
fig_perf.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig_perf.show()

In [ ]:
# Cell 7: 多維統計指標與最近交易明細
rm = RiskManager(trades, strat_equity)
metrics = rm.calculate_metrics()

print("=== 績效與風險統計指標 ===")
for key, val in metrics.items():
    if isinstance(val, float):
        if key in ["CAGR", "Max Drawdown", "Ulcer Index", "Win Rate", "Expectancy"]:
            print(f"{key:25s}: {val * 100:.2f}%")
        else:
            print(f"{key:25s}: {val:.2f}")
    else:
        print(f"{key:25s}: {val}")

print("\n=== 最近 10 筆平倉交易明細 ===")
trade_data = [{
    "Entry Date": t.entry_date.strftime("%Y-%m-%d"),
    "Entry Price": round(t.entry_price, 2),
    "Exit Date": t.exit_date.strftime("%Y-%m-%d") if t.exit_date else "Open",
    "Exit Price": round(t.exit_price, 2) if t.exit_price else "Open",
    "PnL (%)": f"{round(t.pnl_pct * 100, 2)}%",
    "Reason": t.exit_reason
} for t in trades]
pd.DataFrame(trade_data).tail(10)

In [ ]:
# Cell 8: Monte Carlo 風險模擬分析
mc_runs = 5000
mc_trades = 50
mc_seed = 42

mc_results = rm.run_monte_carlo(runs=mc_runs, num_trades=mc_trades, seed=mc_seed)
percentiles = mc_results["percentiles"]

print(f"=== Monte Carlo {mc_runs} 次模擬最大回撤分位數報告 ===")
for pct, val in percentiles.items():
    print(f"{pct:20s}: {val * 100:.2f}%")

# 繪製最大回撤發生機率分佈直方圖
fig_mc = go.Figure()
fig_mc.add_trace(go.Histogram(
    x=mc_results["drawdowns"] * 100,
    nbinsx=50,
    marker=dict(
        color='rgba(200, 50, 50, 0.75)',
        line=dict(color='black', width=0.5)
    )
))
fig_mc.update_layout(title='Monte Carlo 模擬最大回撤機率分佈 (Drawdown Distribution)', xaxis_title='Max Drawdown (%)', yaxis_title='Frequency (Runs)', height=400)
fig_mc.show()

In [ ]:
# Cell 9: 完整交易紀錄明細
opt_trade_list = [{
    "交易序號": idx + 1,
    "方向": t.direction,
    "進場日期": t.entry_date.strftime("%Y-%m-%d"),
    "實際買入價 (含滑價/手續費)": round(t.entry_price, 2),
    "出場日期": t.exit_date.strftime("%Y-%m-%d") if t.exit_date else "未平倉",
    "實際賣出價 (含滑價/手續費)": round(t.exit_price, 2) if t.exit_price else "未平倉",
    "單筆損益 (%)": f"{round(t.pnl_pct * 100, 2)}%",
    "離場原因": t.exit_reason
} for idx, t in enumerate(trades)]

df_opt_trades = pd.DataFrame(opt_trade_list)
print(f"=== 參數組合 ST({period}, {multiplier}) 的完整交易紀錄 (共 {len(df_opt_trades)} 筆) ===")
df_opt_trades

In [ ]:
# Cell 10: 基於最大虧損交易設定 Stop Loss 並重新回測
# --- 分析歷史最大虧損交易，建議合理的 Stop Loss 百分比 ---

print("=== 歷史交易虧損分析 (依損益排序) ===")
loss_trades = [t for t in trades if t.pnl_pct < 0]
loss_trades_sorted = sorted(loss_trades, key=lambda x: x.pnl_pct)

print(f"總虧損交易筆數: {len(loss_trades)} / {len(trades)} (虧損率: {len(loss_trades)/len(trades)*100:.1f}%)")
print("\n--- 最大虧損前 10 筆 ---")
for i, t in enumerate(loss_trades_sorted[:10]):
    print(f"  {i+1}. {t.entry_date.strftime('%Y-%m-%d')} ~ {t.exit_date.strftime('%Y-%m-%d') if t.exit_date else 'Open'}: "
          f"PnL={t.pnl_pct*100:.2f}%, Reason={t.exit_reason}, Hold={(t.exit_date - t.entry_date).days if t.exit_date else 'N/A'} 天")

# 統計分布
loss_pcts = [t.pnl_pct * 100 for t in loss_trades]
import numpy as np
print(f"\n虧損分布統計:")
print(f"  平均虧損: {np.mean(loss_pcts):.2f}%")
print(f"  中位數虧損: {np.median(loss_pcts):.2f}%")
print(f"  標準差: {np.std(loss_pcts):.2f}%")
print(f"  最大虧損: {np.min(loss_pcts):.2f}%")
print(f"  25% 分位數: {np.percentile(loss_pcts, 25):.2f}%")
print(f"  75% 分位數: {np.percentile(loss_pcts, 75):.2f}%")
print(f"  90% 分位數: {np.percentile(loss_pcts, 90):.2f}%")
print(f"  95% 分位數: {np.percentile(loss_pcts, 95):.2f}%")

# --- 使用者設定 Stop Loss ---
print("\n" + "="*60)
print(">>> 請根據上述分析設定 Stop Loss 百分比 (建議參考 75%~95% 分位數)")
stop_loss_pct = 0.1  # 例: 8% 停損，請修改此值
print(f"設定 Stop Loss: {stop_loss_pct*100:.1f}%")

# --- 重新跑回測 (含 Stop Loss) ---
print("\n=== 重新回測 (含 Stop Loss) ===")
engine_sl = BacktestEngine(df_indicators)
trades_sl, strat_equity_sl, bench_equity_sl = engine_sl.run(
    long_only=True,
    stop_loss_pct=stop_loss_pct,      # 啟用停損
    take_profit_pct=0.0,
    commission_bps=10.0,
    slippage_bps=5.0,
    position_sizing_mode="fixed_fraction"
)

# --- 績效比較 ---
rm_orig = RiskManager(trades, strat_equity)
rm_sl = RiskManager(trades_sl, strat_equity_sl)

metrics_orig = rm_orig.calculate_metrics()
metrics_sl = rm_sl.calculate_metrics()

print("\n=== 績效比較：無停損 vs 有停損 ===")
comparison_keys = ["CAGR", "Max Drawdown", "Ulcer Index", "Sharpe Ratio", "Sortino Ratio", 
                   "Win Rate", "Profit Factor", "Calmar Ratio", "Expectancy", "Avg Holding Days", "Annual Trades"]

for key in comparison_keys:
    v1 = metrics_orig.get(key, 0)
    v2 = metrics_sl.get(key, 0)
    if isinstance(v1, float) and isinstance(v2, float):
        if key in ["CAGR", "Max Drawdown", "Ulcer Index", "Win Rate", "Expectancy"]:
            print(f"  {key:20s}: {v1*100:>8.2f}% → {v2*100:>8.2f}%  (Δ={ (v2-v1)*100:+.2f}%)")
        else:
            print(f"  {key:20s}: {v1:>8.2f} → {v2:>8.2f}  (Δ={v2-v1:+.2f})")
    else:
        print(f"  {key:20s}: {v1} → {v2}")

# 交易統計比較
print(f"\n交易筆數: {len(trades)} → {len(trades_sl)} (Δ={len(trades_sl)-len(trades):+d})")

# 止損觸發統計
sl_triggered = [t for t in trades_sl if t.exit_reason == "StopLoss"]
print(f"止損觸發筆數: {len(sl_triggered)} / {len(trades_sl)} ({len(sl_triggered)/len(trades_sl)*100:.1f}%)" if trades_sl else "無交易")

# --- 繪製淨值曲線比較 ---
roll_max_orig = strat_equity.cummax()
dd_orig = (strat_equity - roll_max_orig) / roll_max_orig

roll_max_sl = strat_equity_sl.cummax()
dd_sl = (strat_equity_sl - roll_max_sl) / roll_max_sl

fig_sl = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.7, 0.3])

# 淨值對比
fig_sl.add_trace(go.Scatter(x=strat_equity.index, y=strat_equity, name='無停損', line=dict(color='blue', width=2)), row=1, col=1)
fig_sl.add_trace(go.Scatter(x=strat_equity_sl.index, y=strat_equity_sl, name=f'停損 {stop_loss_pct*100:.0f}%', line=dict(color='orange', width=2)), row=1, col=1)
fig_sl.add_trace(go.Scatter(x=bench_equity.index, y=bench_equity, name='Buy & Hold', line=dict(color='gray', width=1.5, dash='dot')), row=1, col=1)

# 回撤軌跡
fig_sl.add_trace(go.Scatter(x=dd_orig.index, y=dd_orig*100, name='無停損回撤', fill='tozeroy', line=dict(color='rgba(0,0,255,0.3)', width=1)), row=2, col=1)
fig_sl.add_trace(go.Scatter(x=dd_sl.index, y=dd_sl*100, name=f'停損回撤', fill='tozeroy', line=dict(color='rgba(255,165,0,0.3)', width=1)), row=2, col=1)

fig_sl.update_layout(title=f'停損效果比較: ST({period}, {multiplier}) - Stop Loss {stop_loss_pct*100:.0f}%', height=600)
fig_sl.update_yaxes(title_text='Portfolio Value ($)', row=1, col=1)
fig_sl.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig_sl.show()